# Solicitation Analysis

Load SAM.gov opportunity data, prepare it for analysis, and run the solicitation analyzer.

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", None)

DATA_DIR = Path("data")
ATTACHMENTS_DIR = DATA_DIR / "attachments"

## Load opportunities into a DataFrame

In [2]:
with open(DATA_DIR / "opportunities.json") as f:
    opportunities_raw = json.load(f)

print(f"Loaded {len(opportunities_raw)} opportunities")

Loaded 699 opportunities


In [3]:
# Flatten the JSON into a DataFrame, keeping nested fields as-is for now
df = pd.json_normalize(opportunities_raw)
df["postedDate"] = pd.to_datetime(df["postedDate"])
df["responseDeadLine"] = pd.to_datetime(df["responseDeadLine"], utc=True, errors="coerce")

print(f"{len(df)} rows, {len(df.columns)} columns")
df[["noticeId", "title", "type", "postedDate", "naicsCode", "office"]].head(10)

699 rows, 59 columns


,noticeId,title,type,postedDate,naicsCode,office
0,f7d28666b4094675a00c73f97ac50fda,Multiple Award Schedule,Award Notice,2025-05-30,541690,GENERAL SERVICES ADMINISTRATION.FEDERAL ACQUISITION SERVICE.GSA/FAS FURNITURE SYSTEMS MGT DIV
1,f4ebc2c5bd044d3796ff20740ef8ad28,Multiple Award Schedule,Award Notice,2025-05-30,541519,GENERAL SERVICES ADMINISTRATION.FEDERAL ACQUISITION SERVICE.GSA/FAS FURNITURE SYSTEMS MGT DIV
2,da5551b39d834c4d9a96eb674d2129c2,"Replace Leaking Water Pipelines for HVAC system, Camp Walker",Award Notice,2025-05-30,NaN,DEPT OF DEFENSE.DEPT OF THE ARMY.AMC.ACC.ACC-OO.411TH CSB.0906 AQ CO DET A CONTRACTI
3,d38f17fe44eb441ea39bd109a162c4eb,"Housing, Mechanical",Solicitation,2025-05-30,336413,"DEPT OF DEFENSE.DEFENSE LOGISTICS AGENCY.DLA AVIATION.DLA AVIATION OKLAHOMA CITY.DLA AVIATION AT OKLAHOMA CITY, OK"
4,ce44bc9446fe44d88c862aec280ae0bb,Multiple Award Schedule,Award Notice,2025-05-30,541519,GENERAL SERVICES ADMINISTRATION.FEDERAL ACQUISITION SERVICE.GSA/FAS FURNITURE SYSTEMS MGT DIV
5,c97bb61189184575b2d8729eaf83fdc1,Denison Powerhouse Janitorial Services,Award Notice,2025-05-30,561720,DEPT OF DEFENSE.DEPT OF THE ARMY.US ARMY CORPS OF ENGINEERS.ENGINEER DIVISION SOUTHWESTERN.ENDIST TULSA.W076 ENDIST TULSA
6,c581ea48954e4fbdacca9263889b0d8b,Refuse and Recycling Services Multi-site OH105,Combined Synopsis/Solicitation,2025-05-30,562111,DEPT OF DEFENSE.DEPT OF THE ARMY.AMC.ACC.MISSION INSTALLATION CONTRACTING COMMAND.419TH CSB.W6QM MICC FT MCCOY (RC)
7,bc24cf33e0e44577a0ce2d9358a226d9,NSN2935-01-202-5339_CoolerLubricating_C-130_PN8427707_FD2030-25-01972,Sources Sought,2025-05-30,NaN,"DEPT OF DEFENSE.DEFENSE LOGISTICS AGENCY.DLA AVIATION.DLA AVIATION OKLAHOMA CITY.DLA AVIATION AT OKLAHOMA CITY, OK"
8,bb3b57abcb6c40ee8238c3bca9f72b4c,"Noun_INTERFACE UNIT,COMM_Application_B-1_NSN_5895-01-589-9678_Part_Number_544R727G01",Sources Sought,2025-05-30,336413,"DEPT OF DEFENSE.DEFENSE LOGISTICS AGENCY.DLA AVIATION.DLA AVIATION OKLAHOMA CITY.DLA AVIATION AT OKLAHOMA CITY, OK"
9,ba32453fba1a438ca5c369994338a428,"Replace Leaking Water Pipelines for HVAC system, Camp Walker",Award Notice,2025-05-30,NaN,DEPT OF DEFENSE.DEPT OF THE ARMY.AMC.ACC.ACC-OO.411TH CSB.0906 AQ CO DET A CONTRACTI


In [4]:
df["type"].value_counts()

type
Award Notice                      181
Combined Synopsis/Solicitation    179
Solicitation                      162
Presolicitation                    91
Sources Sought                     53
Special Notice                     25
Justification                       8
Name: count, dtype: int64

## Filter to solicitation types only

Keep only the opportunity types we care about for analysis. Delete attachments on disk that belong to excluded rows.

In [5]:
KEEP_TYPES = {"Combined Synopsis/Solicitation", "Solicitation", "Presolicitation"}

# Identify rows to drop and collect their attachment filenames for cleanup
df_excluded = df[~df["type"].isin(KEEP_TYPES)]
excluded_files = []
for files in df_excluded["downloadedFiles"]:
    if isinstance(files, list):
        excluded_files.extend(files)

# Delete attachment files belonging to excluded opportunity types
deleted = 0
for filename in excluded_files:
    path = ATTACHMENTS_DIR / filename
    if path.exists():
        path.unlink()
        deleted += 1

print(f"Removed {deleted} attachment files from excluded types")

# Filter DataFrame to only the types we care about
df = df[df["type"].isin(KEEP_TYPES)].copy().reset_index(drop=True)
print(f"{len(df)} opportunities remaining after filtering to: {KEEP_TYPES}")
df["type"].value_counts()

Removed 0 attachment files from excluded types
432 opportunities remaining after filtering to: {'Combined Synopsis/Solicitation', 'Solicitation', 'Presolicitation'}


type
Combined Synopsis/Solicitation    179
Solicitation                      162
Presolicitation                    91
Name: count, dtype: int64

## Resolve attachment file paths

Map each opportunity's `downloadedFiles` list to absolute paths in `data/attachments/` and filter to files that actually exist on disk.

In [7]:
from spotting_concrete_boats.documents import can_process, compute_attachment_stats


def resolve_attachment_paths(downloaded_files):
    """Return list of existing, processable attachment paths for an opportunity."""
    if not isinstance(downloaded_files, list):
        return []
    paths = []
    for filename in downloaded_files:
        path = ATTACHMENTS_DIR / filename
        if path.exists() and can_process(path):
            paths.append(str(path))
    return paths


df["attachment_paths"] = df["downloadedFiles"].apply(resolve_attachment_paths)
df["attachment_count"] = df["attachment_paths"].apply(len)

# Precompute page counts and file type breakdowns (opens each PDF once)
compute_attachment_stats(df)

print(f"Opportunities with attachments on disk: {(df['attachment_count'] > 0).sum()}")
print(f"Total processable attachments: {df['attachment_count'].sum()}")
print(f"Total PDF pages across all attachments: {df['total_pages'].sum()}")

Opportunities with attachments on disk: 4
Total processable attachments: 9
Total PDF pages across all attachments: 165


## Prep for analysis

Filter to opportunities that have either a meaningful description or downloadable attachments — these are the ones worth running through the analyzer.

In [8]:
from spotting_concrete_boats.documents import build_solicitation_context

# Build structured context for each opportunity from its metadata
df["context"] = df.apply(build_solicitation_context, axis=1)

# Flag opportunities that have substantive content to analyze
df["has_attachments"] = df["attachment_count"] > 0
df["analyzable"] = (df["context"].str.len() > 0) | df["has_attachments"]

print(f"Analyzable opportunities: {df['analyzable'].sum()} / {len(df)}")
print(f"  - With attachments: {df['has_attachments'].sum()}")
print(f"\nSample context:\n{'=' * 80}")
print(df["context"].iloc[0])

Analyzable opportunities: 6 / 6
  - With attachments: 4

Sample context:
Title: Housing, Mechanical
Solicitation Number: SPRTA125Q0349
Notice Type: Solicitation
Agency: DEPT OF DEFENSE.DEFENSE LOGISTICS AGENCY.DLA AVIATION.DLA AVIATION OKLAHOMA CITY.DLA AVIATION AT OKLAHOMA CITY, OK
NAICS Code: 336413
Classification Code: 1650
Posted: 2025-05-30
Response Deadline: 2025-06-30
Response Window: 31 days
Attachments: 1
Total Pages (PDFs): 5

Description:
Noun: Housing, Mechanical; NSN: 1650010539304; P/N: 2009092-1; AMC: 4P


In [9]:
df_to_analyze = df[df["analyzable"]].copy().reset_index(drop=True)
print(f"{len(df_to_analyze)} opportunities ready for analysis")
df_to_analyze[["noticeId", "title", "type", "attachment_count"]].head(10)

6 opportunities ready for analysis


,noticeId,title,type,attachment_count
0,d38f17fe44eb441ea39bd109a162c4eb,"Housing, Mechanical",Solicitation,1
1,c581ea48954e4fbdacca9263889b0d8b,Refuse and Recycling Services Multi-site OH105,Combined Synopsis/Solicitation,0
2,7f67293fee1b4cc5b3ee991ee9c14e4b,Animal Packing Services for the Middle Fork area o,Combined Synopsis/Solicitation,5
3,7f476063a52b4195a17e00495d95e006,AMENDMENT Q402--FY25 12/1 (HOLD) IDIQ Brookfield of Cascadia 648 - OREGON,Solicitation,1
4,621a8143a79b45809c691eb892672c00,Range Classroom Furniture,Combined Synopsis/Solicitation,0
5,3b861f15a52b4194a54b04cf282821a0,"Seal, Air, Aircraft G",Solicitation,2


## Run the analyzer

Iterate over the filtered opportunities and run the `SolicitationAnalyzer` on each one. Results are collected into a list and merged back into the DataFrame.

In [10]:
from spotting_concrete_boats.analyzer import (
    SolicitationAnalyzer,
    results_to_dataframe,
    evidence_to_dataframe,
)

analyzer = SolicitationAnalyzer()
print(analyzer)
analyzer.describe_prompts()

SolicitationAnalyzer(model='claude-sonnet-4-6', prompts=[metadata, sin__compliance_over_outcomes, sin__insider_rewards, sin__requirement_sprawl, sin__rule_pools, virtue__competition_accessibility, virtue__outcome_definition, virtue__structural_fit])


,name,description
0,metadata,Solicitation metadata extraction.
1,sin__compliance_over_outcomes,Sin #2: Compliance Over Outcomes.
2,sin__insider_rewards,Sin #4: Insider Rewards Program.
3,sin__requirement_sprawl,Sin #1: Requirement Sprawl.
4,sin__rule_pools,Sin #3: Rule Pools.
5,virtue__competition_accessibility,Virtue #2: Competition Accessibility.
6,virtue__outcome_definition,Virtue #1: Outcome Definition.
7,virtue__structural_fit,Virtue #3: Structural Fit.


In [ ]:
# Run on a single row — change `row_idx` to pick which one
row_idx = 5
row = df_to_analyze.iloc[row_idx]

print(f"[{row_idx + 1}/{len(df_to_analyze)}] {row['title']}")

file_paths = row["attachment_paths"]
context = row["context"] if row["context"] else None

single_result = analyzer.analyze_from_files(file_paths, description=context)
print(single_result)


In [ ]:
results_to_dataframe(single_result)

,prompt,prompt_type,score,score_label,reasoning,evidence_count
0,metadata,NaN,NaN,NaN,NaN,NaN
1,sin__compliance_over_outcomes,sin,1.0,Minimal,"This document is a solicitation amendment (Amendment 0001 to solicitation 36C26025R0037), not the full solicitation or its attached Performance Work Statement. The amendment contains only two substantive changes: (1) a revision to the evaluation factor language regarding how prices are determined (GEC/VACO publishing Medicare Benchmark rates annually), and (2) a revision to the PWS paragraph 4 Rate Basis language describing how VA will use Medicaid rates with upward adjustments to determine contract rates. There is no QA surveillance plan, no reporting requirements, no meeting cadences, no performance standards, and no accountability mechanisms visible in this two-page document. The underlying solicitation and full PWS would be necessary to meaningfully assess whether the contract's accountability structure emphasizes compliance over outcomes. Without access to those documents, there is insufficient content to evaluate this sin.",0.0
2,sin__insider_rewards,sin,1.0,Minimal,"This document is an amendment (Amendment 0001) to solicitation 36C26025R0037, not the full solicitation itself. The amendment contains only two substantive changes: (1) a revision to evaluation factor language regarding how prices are determined (GEC/VACO Medicare Benchmark rates), and (2) a revision to the PWS paragraph 4 (Rate Basis) regarding how VA will state Medicaid rates. There are no experience requirements, evaluation criteria for vendor qualifications, past performance standards, personnel qualification requirements, or other content that would allow a meaningful assessment of insider rewards. The underlying solicitation, PWS, and evaluation criteria documents are not included. Without access to the full solicitation package — particularly the evaluation criteria, experience requirements, and performance work statement — it is not possible to assess whether the solicitation contains experience requirements that effectively exclude non-incumbent vendors.",0.0
3,sin__requirement_sprawl,sin,1.0,Minimal,"This posting is an amendment (Amendment 0001) to solicitation 36C26025R0037, not the base solicitation itself. The document contains only two pages: a standard SF-30 amendment form and a single-page continuation sheet describing two narrow changes — (1) revising the price evaluation factor language in E.13 to clarify that rates are determined by GEC/VACO and published annually as Medicare Benchmark rates, and (2) revising PWS paragraph 4 to update the Rate Basis language regarding Medicaid rates with upward adjustments. The underlying PWS, base solicitation, and full statement of work are not included in this posting. Without access to the base solicitation and complete PWS, there is insufficient content to meaningfully assess whether the solicitation exhibits requirement sprawl. The two amended provisions address rate-setting methodology — a legitimate constraint in a healthcare IDIQ context — and do not alone provide a sufficient window into the overall requirement structure to render a judgment on sprawl.",0.0
4,sin__rule_pools,sin,1.0,Minimal,"This document is a solicitation amendment (Amendment 0001) to solicitation 36C26025R0037, not the base solicitation itself. The amendment contains only two substantive changes: (1) a revision to Factor 3 pricing language clarifying that prices are determined by GEC/VACO via annual Medicare Benchmark rates, and (2) a revision to PWS paragraph 4 (Rate Basis) clarifying how VA will use Medicaid rates with upward adjustments. There is no regulatory citations list, no 'Applicable Documents' section, no compliance framework enumeration, and no SOW/PWS in full. The two-page document does not contain sufficient content to assess the presence or absence of rule pools. The base solicitation and full PWS would be necessary to evaluate this sin meaningfully.",0.0
5,virtue__competi

In [11]:
all_results = []

for idx, row in df_to_analyze.iterrows():
    print(f"\n[{idx + 1}/{len(df_to_analyze)}] {row['title']}")

    file_paths = row["attachment_paths"]
    context = row["context"] if row["context"] else None

    try:
        results = analyzer.analyze_from_files(file_paths, description=context)
        all_results.append({"noticeId": row["noticeId"], "results": results})
    except Exception as e:
        print(f"  Error: {e}")
        all_results.append({"noticeId": row["noticeId"], "results": None, "error": str(e)})

print(f"\nDone. {sum(1 for r in all_results if r['results'])} / {len(all_results)} succeeded.")


[1/3] Housing, Mechanical


Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/macklinfluehr/.local/share/uv/python/cpython-3.12.9-macos-x86_64-none/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x110025480> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/macklinfluehr/.local/share/uv/python/cpython-3.12.9-macos-x86_64-none/lib/python3.12/asyncio/events.py", line 88, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x110025480> is already entered
Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/Users/macklinfluehr/.local/share/uv/python/cpython-3.12.9-macos-x86_64-none/lib/python3.12/asyncio/events.py"

Analysis complete: 5/5 prompts succeeded.

[2/3] Refuse and Recycling Services Multi-site OH105 
Analysis complete: 5/5 prompts succeeded.

[3/3] Animal Packing Services for the Middle Fork area o
Analysis complete: 5/5 prompts succeeded.

Done. 3 / 3 succeeded.


In [ ]:
# View summary for the last successful result
for entry in reversed(all_results):
    if entry["results"]:
        print(f"Result for: {entry['noticeId']}")
        display(results_to_dataframe(entry["results"]))
        break

## View results

In [ ]:
# Build summary and evidence DataFrames for the first successful result as a sanity check
for entry in all_results:
    if entry["results"]:
        print(f"Sample result for: {entry['noticeId']}")
        display(results_to_dataframe(entry["results"]))
        display(evidence_to_dataframe(entry["results"]))
        break

## Load results from batch run

Load per-solicitation result JSON files saved by `scripts/run_analysis.py` or `scripts/run_batch.py`.

In [ ]:
RESULTS_DIR = DATA_DIR / "results"

if RESULTS_DIR.exists():
    saved_results = []
    for result_file in sorted(RESULTS_DIR.glob("*.json")):
        with open(result_file) as f:
            saved_results.append(json.load(f))

    print(f"Loaded {len(saved_results)} saved results from {RESULTS_DIR}")

    # Build combined summary DataFrame
    if saved_results:
        summary_rows = []
        for entry in saved_results:
            if entry.get("results"):
                row_summary = results_to_dataframe(entry["results"])
                row_summary["notice_id"] = entry["notice_id"]
                row_summary["title"] = entry.get("title", "")
                summary_rows.append(row_summary)

        if summary_rows:
            df_summary = pd.concat(summary_rows, ignore_index=True)
            print(f"Combined summary: {len(df_summary)} rows across {len(summary_rows)} solicitations")
            display(df_summary.head(20))
else:
    print(f"No results directory found at {RESULTS_DIR}. Run scripts/run_analysis.py first.")